In [ ]:
import torch
import numpy as np
import pandas as pd
import pickle
import json
import os
import librosa
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


class DataConfig:
    
    DATA_TYPE = 'audio' 

    AUDIO_DATASET_PATH = r'.....'  # Path to GTZAN dataset folder where .wav files are stored
    SAMPLE_RATE = 22050            # Audio sample rate
    AUDIO_PARAMS = {
        'num_mfcc': 40,
        'n_fft': 2048,
        'hop_length': 512,
        'num_segment': 10,
        'target_time_steps': 130
    }
    
    # Common settings
    SAVE_DIR = r'.......'   #Path to folder where you wish to store the preprocessed data, usually found under /genre_original
    TEST_SIZE = 0.2
    VAL_SIZE = 0.1      # 10% of training data for validation
    RANDOM_STATE = 42
    SHUFFLE = True


class AudioProcessor:
    
    def __init__(self, dataset_path, sample_rate=22050):
        self.dataset_path = dataset_path
        self.sample_rate = sample_rate
        self.class_names = None
        
    def get_class_names(self):
        
        if not os.path.exists(self.dataset_path):
            raise FileNotFoundError(f"Audio dataset path not found: {self.dataset_path}")
        
        self.class_names = sorted([d for d in os.listdir(self.dataset_path) 
                                  if os.path.isdir(os.path.join(self.dataset_path, d))])
        if not self.class_names:
            raise ValueError(f"No genre folders found in {self.dataset_path}")
        
        return self.class_names
    
    def extract_features(self, num_mfcc=40, n_fft=2048, hop_length=512, 
                        num_segment=10, target_time_steps=130):
        
        # Get class names
        self.class_names = self.get_class_names()
        print(f"Found {len(self.class_names)} genres: {self.class_names}")
        
        data = {"features": [], "labels": []}
        samples_per_segment = int(self.sample_rate * 30 / num_segment)
        
        for label_idx, genre in enumerate(self.class_names):
            genre_path = os.path.join(self.dataset_path, genre)
            
            # Get all WAV files
            audio_files = sorted([f for f in os.listdir(genre_path) 
                                if f.endswith('.wav')])
            
            if not audio_files:
                print(f"Warning: No .wav files found in {genre_path}")
                continue
            
            print(f"\nProcessing '{genre}' ({len(audio_files)} files):")
            
            for filename in tqdm(audio_files, desc=f"  {genre}"):
                file_path = os.path.join(genre_path, filename)
                
                try:
                    # Load audio
                    y, sr = librosa.load(file_path, sr=self.sample_rate)
                    
                    # Ensure exactly 30 seconds
                    target_length = self.sample_rate * 30
                    if len(y) < target_length:
                        y = np.pad(y, (0, target_length - len(y)), mode='constant')
                    else:
                        y = y[:target_length]
                    
                    # Process segments
                    for n in range(num_segment):
                        start = samples_per_segment * n
                        end = samples_per_segment * (n + 1)
                        segment = y[start:end]
                        
                        # Extract MFCCs
                        mfcc = librosa.feature.mfcc(
                            y=segment,
                            sr=sr,
                            n_mfcc=num_mfcc,
                            n_fft=n_fft,
                            hop_length=hop_length
                        )
                        
                        # Delta and Delta-Delta features
                        mfcc_delta = librosa.feature.delta(mfcc)
                        mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
                        
                        # Stack features
                        features = np.stack([mfcc, mfcc_delta, mfcc_delta2], axis=-1)
                        
                        # Pad/truncate time dimension
                        if features.shape[1] < target_time_steps:
                            pad_width = ((0, 0), 
                                       (0, target_time_steps - features.shape[1]), 
                                       (0, 0))
                            features = np.pad(features, pad_width, mode='constant')
                        else:
                            features = features[:, :target_time_steps, :]
                        
                        data["features"].append(features)
                        data["labels"].append(label_idx)
                        
                except Exception as e:
                    # If error, skip this file and continue
                    print(f"Skipping {filename}: {str(e)[:50]}...")
                    continue
        
        # Check if we have any data
        if not data["features"]:
            raise ValueError("No audio features extracted. Check dataset path and files.")
        
        # Convert to numpy arrays
        X = np.array(data["features"], dtype=np.float32)  # Shape: (N, n_mfcc, time, 3)
        y = np.array(data["labels"], dtype=np.int64)
        
        print(f"\n Audio feature extraction complete!")
        print(f"   Total samples: {X.shape[0]}")
        print(f"   Feature shape: {X.shape[1:]}")  # (n_mfcc, time, 3)
        print(f"   Labels shape: {y.shape}")
        
        return X, y, self.class_names


def load_and_preprocess_data(config):

    
    if config.DATA_TYPE == 'audio':
        print(" Processing audio data...")
        
        # Initialize audio processor
        audio_processor = AudioProcessor(
            dataset_path=config.AUDIO_DATASET_PATH,
            sample_rate=config.SAMPLE_RATE
        )
        
        # Extract audio features
        X, y, class_names = audio_processor.extract_features(
            num_mfcc=config.AUDIO_PARAMS['num_mfcc'],
            n_fft=config.AUDIO_PARAMS['n_fft'],
            hop_length=config.AUDIO_PARAMS['hop_length'],
            num_segment=config.AUDIO_PARAMS['num_segment'],
            target_time_steps=config.AUDIO_PARAMS['target_time_steps']
        )
        
        # Store original shape and class info
        original_shape = X.shape
        num_classes = len(class_names)
        feature_names = [f'mfcc_{i}' for i in range(config.AUDIO_PARAMS['num_mfcc'])]
        
    else:
        # Tabular data processing (remains as before if you need it)
        raise ValueError("Only audio data type is currently supported in this version")
    
    # Split into train/val/test
    print("\n  Splitting data...")
    
    # First split: train+val vs test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, 
        test_size=config.TEST_SIZE, 
        random_state=config.RANDOM_STATE,
        shuffle=config.SHUFFLE,
        stratify=y if len(np.unique(y)) > 1 else None
    )
    
    # Second split: train vs val
    val_ratio = config.VAL_SIZE / (1 - config.TEST_SIZE)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_ratio,
        random_state=config.RANDOM_STATE,
        shuffle=config.SHUFFLE,
        stratify=y_temp if len(np.unique(y_temp)) > 1 else None
    )
    
    print(f"   Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
    print(f"   Val set:   {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
    print(f"   Test set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
    
    # Convert to PyTorch tensors
    print("\n Converting to PyTorch tensors...")
    
    # For audio data: permute to (N, channels, height, width) format
    # Original shape: (N, n_mfcc, time, 3)
    # After permute: (N, 3, n_mfcc, time)
    X_train_tensor = torch.FloatTensor(X_train).permute(0, 3, 1, 2)  # (N, 3, n_mfcc, time)
    X_val_tensor = torch.FloatTensor(X_val).permute(0, 3, 1, 2)
    X_test_tensor = torch.FloatTensor(X_test).permute(0, 3, 1, 2)
    
    y_train_tensor = torch.LongTensor(y_train)
    y_val_tensor = torch.LongTensor(y_val)
    y_test_tensor = torch.LongTensor(y_test)
    
    # Prepare results dictionary
    result = {
        # PyTorch tensors
        'X_train': X_train_tensor,
        'y_train': y_train_tensor,
        'X_val': X_val_tensor,
        'y_val': y_val_tensor,
        'X_test': X_test_tensor,
        'y_test': y_test_tensor,
        
        # NumPy arrays (for compatibility)
        'X_train_np': X_train,
        'y_train_np': y_train,
        'X_val_np': X_val,
        'y_val_np': y_val,
        'X_test_np': X_test,
        'y_test_np': y_test,
        
        # Metadata
        'data_type': config.DATA_TYPE,
        'original_shape': original_shape,
        'num_classes': num_classes,
        'class_names': class_names,
        'feature_names': feature_names,
        
        # For tabular data
        'num_features': X.shape[1] if config.DATA_TYPE == 'tabular' else None,
        
        # Preprocessing info
        'scaler': None,
        'label_encoder': None,
        'train_val_test_split': {
            'train_samples': len(X_train),
            'val_samples': len(X_val),
            'test_samples': len(X_test),
            'test_size': config.TEST_SIZE,
            'val_size': config.VAL_SIZE
        },
        'preprocessing_info': {
            'preprocessing_done': 'train_val_test_split, tensor_conversion',
            'normalization': None
        }
    }
    
    # Add audio-specific info
    if config.DATA_TYPE == 'audio':
        result['audio_params'] = config.AUDIO_PARAMS
        result['sample_rate'] = config.SAMPLE_RATE
        result['num_features'] = config.AUDIO_PARAMS['num_mfcc']
    
    print("\n Preprocessing complete!")
    return result

def save_data_in_formats(data, save_dir, config):
    """Save data in multiple formats"""
    
    # Create save directory
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    print(f"\n Saving data to: {save_path.absolute()}")
    
    # Save metadata
    metadata = {
        'created_at': pd.Timestamp.now().isoformat(),
        'data_info': {
            'data_type': data['data_type'],
            'num_samples_total': data['original_shape'][0],
            'num_features': str(data['num_features']),  # Convert to string for JSON
            'num_classes': data['num_classes'],
            'class_names': data['class_names'],
            'feature_names': data['feature_names']
        },
        'split_info': data['train_val_test_split'],
        'preprocessing_info': data['preprocessing_info'],
        'config': {
            'data_type': config.DATA_TYPE,
            'test_size': config.TEST_SIZE,
            'val_size': config.VAL_SIZE,
            'random_state': config.RANDOM_STATE
        }
    }
    
    # Add audio-specific config
    if config.DATA_TYPE == 'audio':
        metadata['audio_config'] = {
            'dataset_path': config.AUDIO_DATASET_PATH,
            'sample_rate': config.SAMPLE_RATE,
            'audio_params': config.AUDIO_PARAMS
        }
    
    metadata_file = save_path / 'metadata.json'
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    print(f"  Metadata saved: {metadata_file}")
    
    # 1. PyTorch format (.pt) - RECOMMENDED
    torch_data = {
        'X_train': data['X_train'],
        'y_train': data['y_train'],
        'X_val': data['X_val'],
        'y_val': data['y_val'],
        'X_test': data['X_test'],
        'y_test': data['y_test'],
        'class_names': data['class_names'],
        'feature_names': data['feature_names'],
        'data_type': data['data_type']
    }
    
    # Add audio-specific info
    if data['data_type'] == 'audio':
        torch_data['audio_params'] = data['audio_params']
        torch_data['sample_rate'] = data['sample_rate']
    
    torch_file = save_path / 'data_tensors.pt'
    torch.save(torch_data, torch_file)
    print(f" PyTorch tensors saved: {torch_file}")
    
    # 2. Save preprocessing objects separately (if they exist)
    preprocessors = {}
    if data['scaler'] is not None:
        preprocessors['scaler'] = data['scaler']
    if data['label_encoder'] is not None:
        preprocessors['label_encoder'] = data['label_encoder']
    
    if preprocessors:
        preproc_file = save_path / 'preprocessors.pkl'
        with open(preproc_file, 'wb') as f:
            pickle.dump(preprocessors, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f" Preprocessors saved: {preproc_file}")
    
    return save_path

def load_preprocessed_data(load_dir='preprocessed_data', format='torch'):
    load_path = Path(load_dir)
    
    if format == 'torch':
        # Load PyTorch tensors
        data = torch.load(load_path / 'data_tensors.pt')
        
        # Load metadata
        with open(load_path / 'metadata.json', 'r') as f:
            metadata = json.load(f)
        
        # Load preprocessors if they exist
        preprocessors_path = load_path / 'preprocessors.pkl'
        if preprocessors_path.exists():
            with open(preprocessors_path, 'rb') as f:
                preprocessors = pickle.load(f)
            data.update(preprocessors)
        else:
            data.update({'scaler': None, 'label_encoder': None})
        
        data['metadata'] = metadata
        
    else:
        raise ValueError(f"Unsupported format: {format}. Only 'torch' is currently supported.")
    
    print(f" Loaded data from {load_path} in {format} format")
    print(f"   Data type: {data['data_type']}")
    print(f"   Shape: {data['X_train'].shape}")
    
    return data

def create_dataloaders(data, batch_size=32):

    from torch.utils.data import DataLoader, TensorDataset
    
    # Create datasets
    train_dataset = TensorDataset(data['X_train'], data['y_train'])
    val_dataset = TensorDataset(data['X_val'], data['y_val'])
    test_dataset = TensorDataset(data['X_test'], data['y_test'])
    
    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False
    )
    
    return train_loader, val_loader, test_loader

def main():
    config = DataConfig()
    
    try:
        # Step 1: Load and preprocess data
        print("\n" + "=" * 60)
        print("STARTING DATA PREPROCESSING")
        print("=" * 60)
        print(f"Data type: {config.DATA_TYPE}")
        
        if config.DATA_TYPE == 'audio':
            print(f"Audio dataset path: {config.AUDIO_DATASET_PATH}")
        
        preprocessed_data = load_and_preprocess_data(config)
        
        # Step 2: Save in multiple formats
        save_dir = save_data_in_formats(preprocessed_data, config.SAVE_DIR, config)
        
        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(f" Data type: {preprocessed_data['data_type']}")
        print(f" Original shape: {preprocessed_data['original_shape']}")
        print(f" Number of classes: {preprocessed_data['num_classes']}")
        print(f" Class names: {preprocessed_data['class_names']}")
        
        if preprocessed_data['data_type'] == 'audio':
            print(f" Audio feature shape: {preprocessed_data['X_train'].shape[1:]}")
        
        print(f" Saved to: {save_dir.absolute()}")
        
        # Step 3: Demonstrate loading
        print("\n" + "=" * 60)
        print("DEMONSTRATING LOADING")
        print("=" * 60)
        
        # Load data in PyTorch format (recommended)
        loaded_data = load_preprocessed_data(config.SAVE_DIR, format='torch')
        
        # Create DataLoaders
        train_loader, val_loader, test_loader = create_dataloaders(
            loaded_data, batch_size=32
        )
        
        print(f"\n DataLoader sizes:")
        print(f"   Train batches: {len(train_loader)}")
        print(f"   Val batches: {len(val_loader)}")
        print(f"   Test batches: {len(test_loader)}")
        
        # Show a sample batch
        for X_batch, y_batch in train_loader:
            print(f"\n Sample batch:")
            print(f"   X shape: {X_batch.shape}")
            print(f"   y shape: {y_batch.shape}")
            
            if loaded_data['data_type'] == 'audio':
                print(f"   Audio batch info:")
                print(f"     Channels: {X_batch.shape[1]}")
                print(f"     MFCC coefficients: {X_batch.shape[2]}")
                print(f"     Time steps: {X_batch.shape[3]}")
            
            print(f"   y values sample: {y_batch[:5].tolist()}")
            break
        
        print("\n Data preprocessing and storage complete!")
        print(f"You can now use the saved data for training by calling:")
        print(f"   data = load_preprocessed_data('{config.SAVE_DIR}', format='torch')")
        
    except FileNotFoundError as e:
        print(f"\n Error: {e}")
        if config.DATA_TYPE == 'audio':
            print(f"Please ensure the audio dataset folder exists at: {config.AUDIO_DATASET_PATH}")
            print("The folder should contain genre subfolders with .wav files")
        
        print("\nTo use this script:")
        if config.DATA_TYPE == 'audio':
            print("1. Download the GTZAN dataset (or your audio dataset)")
            print("2. Update DataConfig.AUDIO_DATASET_PATH to point to your dataset folder")
            print("3. Ensure folder structure: dataset/genre/*.wav")
        
        print(f"4. Set DataConfig.DATA_TYPE = 'audio'")
    
    except Exception as e:
        print(f"\n Unexpected error: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()